In [1]:
import os
import time
from typing import Tuple

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models


# -----------------------------
# 1. Device setup (M2 optimized)
# -----------------------------
def get_device() -> torch.device:
    """
    Select the best available device.
    Prefer Apple MPS (for M1/M2 GPUs), then CUDA, else CPU.
    """
    if torch.backends.mps.is_available():
        print("✅ Using Apple MPS (Metal) GPU")
        return torch.device("mps")
    elif torch.cuda.is_available():
        print("✅ Using CUDA GPU")
        return torch.device("cuda")
    else:
        print("⚠️ MPS/CUDA not available, using CPU")
        return torch.device("cpu")


# -----------------------------
# 2. Paths and transforms
# -----------------------------
DATA_ROOT = "/Users/sabarish/Desktop/DR_DATA/Eyepacs, Aptos, Messidor Diabetic Retinopathy/augmented_resized_V2"

def get_dataloaders(batch_size: int = 32) -> Tuple[DataLoader, DataLoader, DataLoader, int]:
    """
    Create train/val/test dataloaders from your folder structure.
    Uses standard ImageNet-style transforms.
    """

    # ImageNet mean & std (good default for ResNet)
    imagenet_mean = [0.485, 0.456, 0.406]
    imagenet_std = [0.229, 0.224, 0.225]

    train_transform = transforms.Compose([
        transforms.Resize((256, 256)),
        transforms.RandomResizedCrop(224),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize(mean=imagenet_mean, std=imagenet_std),
    ])

    eval_transform = transforms.Compose([
        transforms.Resize((256, 256)),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize(mean=imagenet_mean, std=imagenet_std),
    ])

    train_dir = os.path.join(DATA_ROOT, "train")
    val_dir = os.path.join(DATA_ROOT, "val")
    test_dir = os.path.join(DATA_ROOT, "test")

    # ImageFolder expects class subfolders: 0,1,2,3,4 (which you already have)
    train_dataset = datasets.ImageFolder(root=train_dir, transform=train_transform)
    val_dataset   = datasets.ImageFolder(root=val_dir,   transform=eval_transform)
    test_dataset  = datasets.ImageFolder(root=test_dir,  transform=eval_transform)

    num_classes = len(train_dataset.classes)
    print(f"✅ Found {num_classes} classes: {train_dataset.classes}")

    # DataLoader: num_workers=4 is safe on Mac; pin_memory not used with MPS
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True,
                              num_workers=4)
    val_loader   = DataLoader(val_dataset,   batch_size=batch_size, shuffle=False,
                              num_workers=4)
    test_loader  = DataLoader(test_dataset,  batch_size=batch_size, shuffle=False,
                              num_workers=4)

    return train_loader, val_loader, test_loader, num_classes


# -----------------------------
# 3. Model definition (Model A)
# -----------------------------
def build_model(num_classes: int, device: torch.device) -> nn.Module:
    """
    Build a ResNet-18 classifier with ImageNet weights and
    replace the final layer for 5-class DR grading.
    """
    weights = models.ResNet18_Weights.IMAGENET1K_V1
    model = models.resnet18(weights=weights)

    # Replace final fully connected layer
    in_features = model.fc.in_features
    model.fc = nn.Linear(in_features, num_classes)

    model = model.to(device)
    return model


# -----------------------------
# 4. Training and evaluation utils
# -----------------------------
def train_one_epoch(
    model: nn.Module,
    dataloader: DataLoader,
    criterion: nn.Module,
    optimizer: torch.optim.Optimizer,
    device: torch.device,
) -> Tuple[float, float]:
    """
    Train model for one epoch.
    Returns: (average_loss, accuracy)
    """
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in dataloader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()

        optimizer.step()

        running_loss += loss.item() * images.size(0)

        _, preds = torch.max(outputs, 1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    epoch_loss = running_loss / total
    epoch_acc = correct / total
    return epoch_loss, epoch_acc


def evaluate(
    model: nn.Module,
    dataloader: DataLoader,
    criterion: nn.Module,
    device: torch.device,
) -> Tuple[float, float]:
    """
    Evaluate model on validation or test set.
    Returns: (average_loss, accuracy)
    """
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0

    # No gradients during evaluation
    with torch.no_grad():
        for images, labels in dataloader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            running_loss += loss.item() * images.size(0)
            _, preds = torch.max(outputs, 1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

    epoch_loss = running_loss / total
    epoch_acc = correct / total
    return epoch_loss, epoch_acc


# -----------------------------
# 5. Main training loop
# -----------------------------
def train_model_a(
    num_epochs: int = 20,
    batch_size: int = 32,
    learning_rate: float = 1e-4,
    weight_decay: float = 1e-4,
    model_save_path: str = "modelA_resnet18_best.pth",
):
    device = get_device()

    # Load data
    train_loader, val_loader, test_loader, num_classes = get_dataloaders(batch_size)

    # Build model
    model = build_model(num_classes, device)

    # Loss and optimizer
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=learning_rate,
        weight_decay=weight_decay,
    )

    best_val_acc = 0.0
    best_model_state = None

    print("🚀 Starting training (Model A: classification only)\n")

    for epoch in range(num_epochs):
        start_time = time.time()

        train_loss, train_acc = train_one_epoch(
            model, train_loader, criterion, optimizer, device
        )
        val_loss, val_acc = evaluate(
            model, val_loader, criterion, device
        )

        epoch_time = time.time() - start_time

        print(
            f"Epoch [{epoch + 1}/{num_epochs}] "
            f"- Time: {epoch_time:.1f}s "
            f"- Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f} "
            f"- Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}"
        )

        # Save best model by validation accuracy
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_model_state = model.state_dict()
            torch.save(best_model_state, model_save_path)
            print(f"💾 New best model saved with val acc = {best_val_acc:.4f}\n")

    print("\n✅ Training complete.")
    print(f"⭐ Best validation accuracy: {best_val_acc:.4f}")
    print(f"📁 Best model weights saved to: {model_save_path}")

    # -----------------------------
    # Final test evaluation
    # -----------------------------
    if best_model_state is not None:
        model.load_state_dict(best_model_state)
    test_loss, test_acc = evaluate(model, test_loader, criterion, device)
    print(f"\n🧪 Final Test Loss: {test_loss:.4f}, Test Acc: {test_acc:.4f}")


if __name__ == "__main__":
    # You can tweak these if needed
    train_model_a(
        num_epochs=20,
        batch_size=32,
        learning_rate=1e-4,
        weight_decay=1e-4,
        model_save_path="modelA_resnet18_best.pth",
    )


✅ Using Apple MPS (Metal) GPU
✅ Found 5 classes: ['0', '1', '2', '3', '4']
🚀 Starting training (Model A: classification only)

Epoch [1/20] - Time: 1192.1s - Train Loss: 0.8650, Train Acc: 0.6695 - Val Loss: 0.7102, Val Acc: 0.7386
💾 New best model saved with val acc = 0.7386

Epoch [2/20] - Time: 1190.0s - Train Loss: 0.7658, Train Acc: 0.7096 - Val Loss: 0.6771, Val Acc: 0.7548
💾 New best model saved with val acc = 0.7548

Epoch [3/20] - Time: 1210.4s - Train Loss: 0.7324, Train Acc: 0.7220 - Val Loss: 0.6812, Val Acc: 0.7484
Epoch [4/20] - Time: 1218.0s - Train Loss: 0.7077, Train Acc: 0.7322 - Val Loss: 0.6207, Val Acc: 0.7749
💾 New best model saved with val acc = 0.7749

Epoch [5/20] - Time: 1239.1s - Train Loss: 0.6901, Train Acc: 0.7389 - Val Loss: 0.6499, Val Acc: 0.7700
Epoch [6/20] - Time: 1360.6s - Train Loss: 0.6802, Train Acc: 0.7443 - Val Loss: 0.6047, Val Acc: 0.7835
💾 New best model saved with val acc = 0.7835

Epoch [7/20] - Time: 1386.2s - Train Loss: 0.6637, Train Ac